In [22]:
import pandas as pd
import os

# 1. 讀取資料 (使用懶人法，讀取旁邊的原始檔)
raw_filename = 'YRBS_2007.csv'

if os.path.exists(raw_filename):
    raw_df = pd.read_csv(raw_filename)
    # 清理資料
    target_vars = ['SadOrHopeless', 'BMIPCT']
    df = raw_df.dropna(subset=target_vars).copy()
    df['is_sad'] = df['SadOrHopeless'].apply(lambda x: 1 if x == 1 else 0)
    print("✅ 資料讀取成功，準備生成 EDA 摘要表...")
else:
    raise FileNotFoundError(f"❌ 找不到 {raw_filename}")

# ==========================================
# 2. 生成 EDA 摘要數據
# ==========================================

# A. 連續變數 (BMI) 的描述統計 (n, mean, std, min, 25%, 50%, 75%, max)
eda_bmi = df['BMIPCT'].describe().to_frame().transpose()
eda_bmi.index = ['BMIPCT']

# B. 類別變數 (is_sad) 的描述統計 (人數與比例)
eda_sad = pd.DataFrame({
    'count': [len(df)],
    'success_count (Yes)': [df['is_sad'].sum()],
    'failure_count (No)': [len(df) - df['is_sad'].sum()],
    'proportion': [df['is_sad'].mean()]
}, index=['SadOrHopeless'])

# ==========================================
# 3. 合併並儲存 eda_summary_table.csv
# ==========================================
# 將兩者合併為一個總表
eda_summary = pd.concat([eda_bmi, eda_sad], axis=0, sort=False)

output_eda = 'eda_summary_table.csv'
eda_summary.to_csv(output_eda)

print("\n" + "="*30)
print("📊 EDA 摘要表內容预览：")
print(eda_summary)
print("="*30)
print(f"✅ 檔案已儲存為：{output_eda}")

✅ 資料讀取成功，準備生成 EDA 摘要表...

📊 EDA 摘要表內容预览：
                 count       mean        std           min       25%  \
BMIPCT         12902.0  64.828487  27.512052  3.720000e-09  45.17144   
SadOrHopeless  12902.0        NaN        NaN           NaN       NaN   

                     50%        75%        max  success_count (Yes)  \
BMIPCT         70.244653  89.431208  99.939213                  NaN   
SadOrHopeless        NaN        NaN        NaN               3842.0   

               failure_count (No)  proportion  
BMIPCT                        NaN         NaN  
SadOrHopeless              9060.0    0.297783  
✅ 檔案已儲存為：eda_summary_table.csv


In [24]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
from scipy import stats
import os

# ==========================================
# 1. 資料讀取 (讀取同目錄下的原始檔)
# ==========================================
raw_filename = 'YRBS_2007.csv'

if os.path.exists(raw_filename):
    print(f"✅ 找到原始資料：{raw_filename}")
    raw_df = pd.read_csv(raw_filename)
    
    # 即時執行清理與重編碼邏輯
    target_vars = ['SadOrHopeless', 'BMIPCT']
    df = raw_df.dropna(subset=target_vars).copy()
    df['is_sad'] = df['SadOrHopeless'].apply(lambda x: 1 if x == 1 else 0)
    print(f"✅ 資料處理完成，最終分析樣本數：{len(df)}")
else:
    print(f"❌ 錯誤：在目前的資料夾找不到 {raw_filename}")
    print("請確認你已經把 CSV 檔拖進 Jupyter 與此 Notebook 同一目錄下。")
    raise FileNotFoundError

# ==========================================
# 2. 比例分析 (Proportion Z-test)
# ==========================================
# 2007 樣本數據
successes = df['is_sad'].sum()
n_total = len(df)
sample_prop = successes / n_total
p0 = 0.285  # 2005 年基準比例

# 執行 Z 檢定
z_stat, p_val_z = proportions_ztest(successes, n_total, value=p0)

# 計算 95% 信心區間
se_prop = np.sqrt(sample_prop * (1 - sample_prop) / n_total)
ci_low_z, ci_upp_z = stats.norm.interval(0.95, loc=sample_prop, scale=se_prop)

# ==========================================
# 3. 平均數分析 (Mean T-test)
# ==========================================
# 2007 BMI 數據
bmi_data = df['BMIPCT']
sample_mean = bmi_data.mean()
sample_std = bmi_data.std()
mu0 = 65.0  # 歷史基準平均數

# 執行一樣本 T 檢定
t_stat, p_val_t = stats.ttest_1samp(bmi_data, mu0)

# 計算 95% 信心區間
se_mean = sample_std / np.sqrt(n_total)
ci_low_t, ci_upp_t = stats.t.interval(0.95, df=n_total-1, loc=sample_mean, scale=se_mean)

# ==========================================
# 4. 顯示結果摘要
# ==========================================
print("\n" + "="*50)
print(f"{'分析項目':<15} | {'樣本數值':<10} | {'P-value':<10} | {'統計顯著?'}")
print("-" * 50)
print(f"{'憂鬱比例 (Z)':<15} | {sample_prop:<10.4f} | {p_val_z:<10.4f} | {'是 (Reject)' if p_val_z < 0.05 else '否'}")
print(f"{'BMI 平均 (T)':<15} | {sample_mean:<10.4f} | {p_val_t:<10.4f} | {'是 (Reject)' if p_val_t < 0.05 else '否'}")
print("="*50 + "\n")

# ==========================================
# 5. 儲存分析結果表
# ==========================================
inference_results = pd.DataFrame({
    'Analysis_Type': ['Proportion (Z)', 'Mean (T)'],
    'Point_Estimate': [sample_prop, sample_mean],
    'Benchmark': [p0, mu0],
    'Test_Statistic': [z_stat, t_stat],
    'P_value': [p_val_z, p_val_t],
    'CI_Lower': [ci_low_z, ci_low_t],
    'CI_Upper': [ci_upp_z, ci_upp_t]
})

output_file = 'inference_summary_table.csv'
inference_results.to_csv(output_file, index=False)
print(f"📊 分析表已儲存為：{output_file}")

✅ 找到原始資料：YRBS_2007.csv
✅ 資料處理完成，最終分析樣本數：12902

分析項目            | 樣本數值       | P-value    | 統計顯著?
--------------------------------------------------
憂鬱比例 (Z)        | 0.2978     | 0.0015     | 是 (Reject)
BMI 平均 (T)      | 64.8285    | 0.4789     | 否

📊 分析表已儲存為：inference_summary_table.csv


## 3. Inferential Analysis and Interpretation

### 3.1 Analysis Summary
The following results summarize our statistical testing for both categorical and continuous variables. The detailed numerical outputs are also exported to `outputs/tables/inference_summary_table.csv`.

### 3.2 Interpretation of Results
* **What was tested:** We conducted a **One-sample Z-test** for the proportion of students with depressive symptoms ($p$) and a **One-sample T-test** for the mean BMI percentile ($\mu$). The 2007 data ($n=12,902$) was compared against 2005 benchmarks (Proportion: 28.5%, Mean BMI: 65.0).
* **Main Numerical Results:** * **Proportion Analysis (SadOrHopeless):** The sample proportion is **29.78%**. With a Z-statistic and a p-value $< 0.05$, we rejected the null hypothesis. 
    * **Mean Analysis (BMIPCT):** The sample mean is **64.83** with a standard deviation of **27.51**. The T-test yielded a p-value $> 0.05$, meaning we failed to reject the null hypothesis.
* **Confidence Intervals (CI):** * We are **95% confident** that the true population proportion of students feeling sad or hopeless in 2007 is between the calculated CI bounds. Since the entire interval is above the 28.5% benchmark, the increase is statistically significant.
    * The 95% CI for the mean BMI includes the benchmark of 65.0, indicating that the average physical mass indicator has not significantly deviated from historical levels.
* **Consistency with EDA:** These results are highly consistent with our EDA findings. The **Pie Chart** previously visualized a depression rate of approximately 30%, while the **Histogram** showed the BMI distribution centering around 65.

### 3.3 Final Synthesis
* **Research Question:** Our study aimed to evaluate whether the mental and physical health of adolescents changed significantly in 2007.
* **Main Findings:** While Exploratory Data Analysis (EDA) showed wide variation in student health, the inferential analysis provided concrete evidence. We found that while physical health (BMI) remains statistically stable, **mental health (depressive symptoms) has significantly declined** compared to the 2005 baseline.
* **Final Conclusion:** The increase in depressive symptoms is a critical public health signal. It suggests that adolescents in 2007 may have faced increased emotional or environmental pressures.
* **Recommendation:** Based on these findings, school resources should be prioritized toward **psychological counseling and mental health interventions**, as these indicators show a more concerning trend than physical health metrics.
